# Preprocessing occupation classification

Clara Krämer, May 2025

### Notebook purpose
- Load classification and aggregate to ISCO 3 level
- To do so, compute classification shares of less granular occupations
- Save to load and match in LFS data preprocessing

In [1]:
import os
import pandas as pd
from pathlib import Path
from data.framework import Esco
from src import utils

# settings
useful_paths = utils.UsefulPaths()
data_dir = Path(useful_paths.data_processed) / "esco"

#### Prepare unweighted shares at all ISCO-08 levels

Read Zaussinger et al. (2025) classifications (ESCO-level)

In [2]:
# 1) Load the new classification
path = data_dir / "ZaussingerSchmidtEgli2025_OccupationClassificationESCOv1.1.csv"
df_cls = pd.read_csv(
    path,
    usecols=["conceptUri", "iscoGroup", "preferredLabel", "occupation_category"]
)

# rename
df_cls = df_cls.rename(columns={
    "iscoGroup": "isco_code",
    "preferredLabel": "preferredLabel_isco",
    "occupation_category": "classification",
})

In [3]:
df_cls

,conceptUri,preferredLabel_isco,classification,isco_code
0,http://data.europa.eu/esco/occupation/00030d09...,technical director,neutral,2654
1,http://data.europa.eu/esco/occupation/1a7fb683...,video and motion picture director,neutral,2654
2,http://data.europa.eu/esco/occupation/2f372afe...,performance lighting director,neutral,2654
3,http://data.europa.eu/esco/occupation/30b25ee4...,animation director,neutral,2654
4,http://data.europa.eu/esco/occupation/3b6bea7d...,video and motion picture producer,neutral,2654
...,...,...,...,...
3003,http://data.europa.eu/esco/occupation/b059d331...,hawker,neutral,9520
3004,http://data.europa.eu/esco/occupation/96a0d8d3...,quick service restaurant crew member,neutral,9411
3005,http://data.europa.eu/esco/occupation/a02a1117...,pizzaiolo,neutral,9411
3006,http://data.europa.eu/esco/occupation/e1cf6897...,kitchen assistant,neutral,9412


Create ESCO-ISCO correspondance file

In [6]:
esco = Esco()
occ = esco.occupations.copy()

# make sure conceptUri is present on both
assert "conceptUri" in occ.columns and "conceptUri" in df_cls.columns

records = []
for n_digits in [1, 2, 3, 4]:
    # pad/truncate isco_code to n_digits
    occ[f"isco_{n_digits}"] = occ["iscoGroup"].str[:n_digits].str.zfill(n_digits)

    # attach ISCO labels
    # e.g. esco.isco_groups has columns ["conceptUri","preferredLabel","code"]
    lookup = esco.isco_groups.rename(columns={"code": f"isco_{n_digits}"})
    lookup = lookup[[f"isco_{n_digits}", "preferredLabel"]].rename(
        columns={"preferredLabel": f"preferredLabel_{n_digits}"}
    )

    # merge classification (ESCO→ISCO) via conceptUri
    df = (
        df_cls
        .merge(occ[["conceptUri", f"isco_{n_digits}"]], on="conceptUri", how="right")
        .merge(lookup, on=f"isco_{n_digits}", how="left")
        .drop(columns=["conceptUri"])
    )

    # count ESCO occupations per ISCO category
    df["n_esco"] = 1
    grp = (
        df
        .groupby(
            [f"isco_{n_digits}",         # ISCO code at this digit
             f"preferredLabel_{n_digits}",   # its label
             "classification"]           # occupation_category
        )["n_esco"]
        .count()
        .reset_index()                            # <-- now grp is a DataFrame
        .rename(columns={
            f"isco_{n_digits}": "isco_code",
            f"preferredLabel_{n_digits}": "preferredLabel_isco"
        })
    )
    grp["isco_level"] = n_digits

    # pivot to wide: one column per classification
    df_piv = grp.pivot_table(
        index=["isco_level", "isco_code", "preferredLabel_isco"],
        columns="classification",
        values="n_esco",
        fill_value=0
    ).reset_index()

    # ensure all five categories exist
    for cat in ["low-carbon", "neutral", "viable-to-decarbonize", "unviable-to-decarbonize"]:
        if cat not in df_piv.columns:
            df_piv[cat] = 0

    # compute totals and shares
    df_piv["N"] = df_piv[["low-carbon", "neutral", "viable-to-decarbonize", "unviable-to-decarbonize"]].sum(axis=1)
    df_piv["share_low-carbon"]                = df_piv["low-carbon"]                 / df_piv["N"]
    df_piv["share_neutral"]                    = df_piv["neutral"]                    / df_piv["N"]
    df_piv["share_viable-to-decarbonize"]      = df_piv["viable-to-decarbonize"]      / df_piv["N"]
    df_piv["share_unviable-to-decarbonize"]    = df_piv["unviable-to-decarbonize"]    / df_piv["N"]
    # high-carbon is viable + unviable
    df_piv["high-carbon"] = df_piv["viable-to-decarbonize"] + df_piv["unviable-to-decarbonize"]
    df_piv["share_high-carbon"] = df_piv["high-carbon"] / df_piv["N"]

    records.append(df_piv)

In [7]:
gbn_shares_final = pd.concat(records, axis=0, ignore_index=True)

utils.save_df_to_files(
    df=gbn_shares_final,
    output_dir=str(data_dir),
    fname_no_ext="final_gbn_shares_by_isco_unweighted_new"
)

In [8]:
gbn_shares_final

classification,isco_level,isco_code,preferredLabel_isco,low-carbon,neutral,unviable-to-decarbonize,viable-to-decarbonize,N,share_low-carbon,share_neutral,share_viable-to-decarbonize,share_unviable-to-decarbonize,high-carbon,share_high-carbon
0,1,0,Armed forces occupations,0.0,21.0,0.0,0.0,21.0,0.000000,1.000000,0.000000,0.000000,0.0,0.000000
1,1,1,Managers,3.0,331.0,2.0,2.0,338.0,0.008876,0.979290,0.005917,0.005917,4.0,0.011834
2,1,2,Professionals,39.0,782.0,14.0,11.0,846.0,0.046099,0.924350,0.013002,0.016548,25.0,0.029551
3,1,3,Technicians and associate professionals,36.0,605.0,11.0,1.0,653.0,0.055130,0.926493,0.001531,0.016845,12.0,0.018377
4,1,4,Clerical support workers,0.0,87.0,2.0,0.0,89.0,0.000000,0.977528,0.000000,0.022472,2.0,0.022472
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
598,4,9613,Sweepers and related labourers,0.0,1.0,0.0,0.0,1.0,0.000000,1.000000,0.000000,0.000000,0.0,0.000000
599,4,9621,"Messengers, package deliverers and luggage por...",0.0,2.0,0.0,0.0,2.0,0.000000,1.000000,0.000000,0.000000,0.0,0.000000
600,4,9622,Odd job persons,0.0,1.0,0.0,0.0,1.0,0.000000,1.000000,0.000000,0.000000,0.0,0.000000
601,4,9623,Meter readers and vending-machine collectors,0.0,2.0,0.0,0.0,2.0,0.000000,1.000000,0.000000,0.000000,0.0,0.000000
